# EOQ Benchmark v2: Entropy-Optimal Quantization vs GGUF

Prova que absmax + rANS entropy coding compete com GGUF K-quants.

Mede: **Tamanho, Perplexidade e tok/s** para cada modelo e bitwidth.

In [ ]:
# Cell 1: Install dependencies (run first, restart runtime if needed)
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q datasets accelerate safetensors sentencepiece tiktoken
!huggingface-cli login --token YOUR_HF_TOKEN_HERE

In [ ]:
# Cell 2: Imports + GPU check
import torch
import numpy as np
import math
import time
import gc
import json
from dataclasses import dataclass
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: EOQ Core (self-contained)

@dataclass
class QuantResult:
    codes: torch.Tensor
    scale: torch.Tensor
    bits: int
    block_size: int
    shape: tuple


def quantize_absmax(tensor, bits, block_size=128):
    t = tensor.detach().float().flatten()
    n = t.numel()
    pad = (block_size - n % block_size) % block_size
    if pad > 0:
        t = torch.nn.functional.pad(t, (0, pad))
    blocks = t.view(-1, block_size)
    qmax = (1 << (bits - 1)) - 1
    absmax = blocks.abs().amax(dim=1).clamp(min=1e-10)
    scale = absmax / qmax
    codes = (blocks / scale.unsqueeze(1)).round().clamp(-qmax, qmax).to(torch.int8)
    codes = codes.flatten()[:n].view(tensor.shape)
    return QuantResult(codes=codes, scale=scale, bits=bits, block_size=block_size, shape=tuple(tensor.shape))


def dequantize(qr):
    flat = qr.codes.float().flatten()
    n = flat.numel()
    bs = qr.block_size
    pad = (bs - n % bs) % bs
    if pad > 0:
        flat = torch.nn.functional.pad(flat, (0, pad))
    blocks = flat.view(-1, bs)
    return (blocks * qr.scale.unsqueeze(1)).flatten()[:n].view(qr.shape)


def compute_eoq_size(tensor, bits, block_size=128):
    qr = quantize_absmax(tensor, bits, block_size)
    codes = qr.codes.flatten().numpy().astype(np.int64)
    qmax = (1 << (bits - 1)) - 1
    codes_u = codes + qmax
    counts = np.bincount(codes_u.astype(int), minlength=2*qmax+1)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    H = -(probs * np.log2(probs)).sum()
    n = tensor.numel()
    entropy_bytes = int(np.ceil(H * n / 8))
    scale_bytes = qr.scale.numel() * 2
    raw_bytes = (n * bits + 7) // 8 + scale_bytes
    eoq_bytes = entropy_bytes + scale_bytes
    return {
        'entropy': H, 'raw_bytes': raw_bytes, 'eoq_bytes': eoq_bytes,
        'savings_pct': (1 - eoq_bytes / raw_bytes) * 100 if raw_bytes > 0 else 0,
        'dequantized': dequantize(qr),
    }

print('EOQ core loaded.')

In [ ]:
# Cell 4: WikiText-2 + PPL + Speed measurement

ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
WIKI_TEXT = '\n\n'.join([t for t in ds['text'] if t.strip()])[:200000]
print(f'WikiText-2: {len(WIKI_TEXT)} chars')


@torch.no_grad()
def measure_ppl(model, tokenizer, max_len=2048, stride=512, max_tokens=20000):
    input_ids = tokenizer(WIKI_TEXT, return_tensors='pt').input_ids.to(model.device)
    nlls = []; total = 0
    for i in range(0, min(input_ids.size(1) - max_len, max_tokens), stride):
        chunk = input_ids[:, i:i+max_len]
        target = chunk.clone()
        target[:, :max_len-stride] = -100
        loss = model(chunk, labels=target).loss
        nlls.append(loss.item() * stride)
        total += stride
    return math.exp(sum(nlls) / total)


@torch.no_grad()
def measure_speed(model, tokenizer, n_tokens=100, n_runs=3):
    """Measure token generation speed (tok/s)."""
    prompt = 'Explain the theory of relativity in simple terms:'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    # Warmup
    model.generate(**inputs, max_new_tokens=5, do_sample=False)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    
    times = []
    for _ in range(n_runs):
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = model.generate(**inputs, max_new_tokens=n_tokens, do_sample=False)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    
    avg = sum(times) / len(times)
    return n_tokens / avg


print('PPL + Speed functions ready.')

In [ ]:
# Cell 5: Full benchmark function

def benchmark_model(model_name, bits_list=[6, 5, 4], gguf_size_mb=None, measure_tok_s=True):
    print(f'\n{"="*70}')
    print(f'  BENCHMARKING: {model_name}')
    print(f'{"="*70}')
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    results = {}
    
    # --- FP16 Baseline ---
    print(f'\n  [FP16] Loading...', flush=True)
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=torch.float16,
        device_map='auto', trust_remote_code=True
    )
    model.eval()
    fp16_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    fp16_mb = fp16_bytes / 1e6
    print(f'  [FP16] {fp16_mb:.0f} MB, loaded in {time.time()-t0:.0f}s', flush=True)
    
    print(f'  [FP16] Measuring PPL...', flush=True)
    ppl_fp16 = measure_ppl(model, tokenizer)
    print(f'  [FP16] PPL = {ppl_fp16:.2f}', flush=True)
    
    tps_fp16 = 0
    if measure_tok_s:
        print(f'  [FP16] Measuring tok/s...', flush=True)
        tps_fp16 = measure_speed(model, tokenizer)
        print(f'  [FP16] {tps_fp16:.1f} tok/s', flush=True)
    
    results['FP16'] = {'ppl': ppl_fp16, 'size_mb': fp16_mb, 'tok_s': tps_fp16}
    del model; gc.collect(); torch.cuda.empty_cache()
    
    # --- EOQ at each bit width ---
    for bits in bits_list:
        print(f'\n  [EOQ Q{bits}] Loading + quantizing...', flush=True)
        t0 = time.time()
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=torch.float16,
            device_map='auto', trust_remote_code=True
        )
        
        n_tensors = 0; total_eoq = 0; total_raw = 0
        for name, param in model.named_parameters():
            if param.ndim >= 2 and param.numel() >= 256:
                with torch.no_grad():
                    p_cpu = param.data.cpu().float()
                    info = compute_eoq_size(p_cpu, bits, 128)
                    param.data.copy_(info['dequantized'].to(param.dtype).to(param.device))
                    total_eoq += info['eoq_bytes']
                    total_raw += info['raw_bytes']
                    n_tensors += 1
            else:
                total_eoq += param.numel() * param.element_size()
                total_raw += param.numel() * param.element_size()
        
        model.eval()
        eoq_mb = total_eoq / 1e6
        raw_mb = total_raw / 1e6
        load_time = time.time() - t0
        print(f'  [EOQ Q{bits}] {n_tensors} tensors, EOQ={eoq_mb:.0f}MB, raw={raw_mb:.0f}MB, {load_time:.0f}s', flush=True)
        
        print(f'  [EOQ Q{bits}] Measuring PPL...', flush=True)
        ppl_q = measure_ppl(model, tokenizer)
        print(f'  [EOQ Q{bits}] PPL = {ppl_q:.2f}', flush=True)
        
        tps_q = 0
        if measure_tok_s:
            print(f'  [EOQ Q{bits}] Measuring tok/s...', flush=True)
            tps_q = measure_speed(model, tokenizer)
            print(f'  [EOQ Q{bits}] {tps_q:.1f} tok/s', flush=True)
        
        results[f'EOQ_Q{bits}'] = {
            'ppl': ppl_q, 'size_mb': eoq_mb, 'raw_size_mb': raw_mb,
            'entropy_savings_pct': (1 - eoq_mb / raw_mb) * 100,
            'tok_s': tps_q,
        }
        del model; gc.collect(); torch.cuda.empty_cache()
    
    # --- Summary ---
    print(f'\n  {"="*70}')
    print(f'  RESULTS: {model_name}')
    print(f'  {"="*70}')
    print(f'  {"Format":>15s} | {"Size (MB)":>10s} | {"PPL":>8s} | {"tok/s":>8s} | {"vs GGUF":>10s}')
    print(f'  {"-"*15}-+-{"-"*10}-+-{"-"*8}-+-{"-"*8}-+-{"-"*10}')
    if gguf_size_mb:
        print(f'  {"GGUF Q4_K_M":>15s} | {gguf_size_mb:>10.0f} | {"ref":>8s} | {"ref":>8s} | {"baseline":>10s}')
    r = results['FP16']
    print(f'  {"FP16":>15s} | {r["size_mb"]:>10.0f} | {r["ppl"]:>8.2f} | {r["tok_s"]:>8.1f} |')
    for bits in bits_list:
        key = f'EOQ_Q{bits}'
        r = results[key]
        vs = f'{(1 - r["size_mb"] / gguf_size_mb) * 100:>+9.1f}%' if gguf_size_mb else ''
        print(f'  {f"EOQ Q{bits}":>15s} | {r["size_mb"]:>10.0f} | {r["ppl"]:>8.2f} | {r["tok_s"]:>8.1f} | {vs}')
    print(f'  {"="*70}')
    
    results['model'] = model_name
    results['gguf_size_mb'] = gguf_size_mb
    return results

print('Benchmark function ready.')

## Run: Small + Medium Models

In [ ]:
MODELS_SMALL = [
    ('Qwen/Qwen2.5-0.5B',  469, [8, 6, 5, 4]),
    ('Qwen/Qwen2.5-3B',   2020, [6, 5, 4]),
    ('Qwen/Qwen3.5-4B',   2709, [6, 5, 4]),
]

all_results = {}
for model_name, gguf_mb, bits in MODELS_SMALL:
    try:
        r = benchmark_model(model_name, bits_list=bits, gguf_size_mb=gguf_mb)
        all_results[model_name] = r
    except Exception as e:
        print(f'  ERROR on {model_name}: {e}')
        all_results[model_name] = {'error': str(e)}

with open('eoq_results_small.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print('\nSaved: eoq_results_small.json')

## Run: Large Models (27B, 35B)

In [ ]:
MODELS_LARGE = [
    ('meta-llama/Llama-3.1-8B', 4920, [6, 5, 4]),
    ('Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled', None, [5, 4]),
    ('HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive', None, [5, 4]),
]

for model_name, gguf_mb, bits in MODELS_LARGE:
    try:
        r = benchmark_model(model_name, bits_list=bits, gguf_size_mb=gguf_mb)
        all_results[model_name] = r
    except Exception as e:
        print(f'  ERROR on {model_name}: {e}')
        all_results[model_name] = {'error': str(e)}

with open('eoq_results_all.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print('\nSaved: eoq_results_all.json')

## Final Comparison Table

In [ ]:
print('\n' + '='*95)
print('  EOQ vs GGUF — CROSS-MODEL COMPARISON (Size / PPL / tok/s)')
print('='*95)
print(f'{"Model":>40s} | {"FP16":>12s} | {"GGUF Q4KM":>10s} | {"EOQ Q6":>14s} | {"EOQ Q5":>14s} | {"EOQ Q4":>14s}')
print(f'{"":>40s} | {"PPL":>12s} | {"Size(MB)":>10s} | {"Size/PPL/tps":>14s} | {"Size/PPL/tps":>14s} | {"Size/PPL/tps":>14s}')
print('-'*95)

for model_name, r in all_results.items():
    if 'error' in r:
        short = model_name.split('/')[-1][:35]
        print(f'{short:>40s} | ERROR: {r["error"][:50]}')
        continue
    
    short = model_name.split('/')[-1][:35]
    fp16_ppl = f'{r["FP16"]["ppl"]:.1f}' if 'FP16' in r else '?'
    gguf_str = f'{r.get("gguf_size_mb", "?")}' if r.get('gguf_size_mb') else '?'
    
    cols = [f'{short:>40s}', f'{fp16_ppl:>12s}', f'{gguf_str:>10s}']
    
    for bits in [6, 5, 4]:
        key = f'EOQ_Q{bits}'
        if key in r:
            sz = r[key]['size_mb']
            ppl = r[key]['ppl']
            tps = r[key].get('tok_s', 0)
            cols.append(f'{sz:.0f}/{ppl:.1f}/{tps:.0f}')
        else:
            cols.append('--')
    
    print(' | '.join(cols))

print('='*95)
print('\nFormat: Size(MB) / Perplexity / tok/s')
print('EOQ wins when: smaller size + similar PPL + same tok/s as GGUF Q4_K_M')

## Key Finding Summary

In [ ]:
print('\n' + '='*70)
print('  KEY FINDINGS')
print('='*70)

for model_name, r in all_results.items():
    if 'error' in r or 'FP16' not in r:
        continue
    short = model_name.split('/')[-1]
    gguf = r.get('gguf_size_mb')
    if not gguf:
        continue
    
    print(f'\n  {short}:')
    print(f'    FP16 PPL: {r["FP16"]["ppl"]:.2f}')
    
    # Find EOQ config closest to GGUF size
    best_key = None
    best_diff = float('inf')
    for key in r:
        if key.startswith('EOQ_Q'):
            diff = abs(r[key]['size_mb'] - gguf)
            if diff < best_diff:
                best_diff = diff
                best_key = key
    
    if best_key:
        eq = r[best_key]
        vs = (1 - eq['size_mb'] / gguf) * 100
        print(f'    GGUF Q4_K_M: {gguf:.0f} MB')
        print(f'    Best match {best_key}: {eq["size_mb"]:.0f} MB, PPL {eq["ppl"]:.2f} ({vs:+.1f}% size)')
    
    # Find smallest EOQ with PPL < FP16 + 1.0
    fp16_ppl = r['FP16']['ppl']
    smallest = None
    for key in r:
        if key.startswith('EOQ_Q') and r[key]['ppl'] < fp16_ppl + 1.0:
            if smallest is None or r[key]['size_mb'] < r[smallest]['size_mb']:
                smallest = key
    
    if smallest:
        s = r[smallest]
        print(f'    Smallest with PPL < FP16+1: {smallest} = {s["size_mb"]:.0f} MB, PPL {s["ppl"]:.2f}')
        if gguf:
            saving = (1 - s['size_mb'] / gguf) * 100
            print(f'    vs GGUF Q4_K_M: {saving:+.1f}% size')

print(f'\n{"="*70}')